# feature engineering 2.0
- following from second meeting with Greenberg 
- feedback included:
-	CARD might be better because we know they have diagnosis 
-	Rerun and make a more specific and comprehensive list of features and ensure that you know what each feature is. 
o	When rerunning if you want to include individual items do it but not domains (not sure what this meant but figure it out and try it)
o	Surprised AQ and Sex not big predictors 
-	Rerun but target group excludes people ‘with diagnosis’ but score below threshold of AQ
-	Rerun with AQ as target Var e.g., 0 class below a 6? and target case 6 and above?
-	Rerun but with my own PCA on each of the questions 
-	Rerun with AQ cut off as target var 
-	Also ‘other’ could have been a text option so maybe look into this?
-	Oh and email don’t use linked in lol 


# notebook uses beginning of feature_engineering 1.0 to set up the same dataset



In [ ]:
import pandas as pd
import numpy as np 
from sklearn.feature_selection import VarianceThreshold
# Add these imports to your notebook
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier

# load matched data 
df = pd.read_csv('data/processed/data_c4_matched_balanced.csv')
import os

# Create the directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)


# 1. feature creation 

# age group bins
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], labels=['0-18', '19-30', '31-45', '46-60', '61+'])

#non linear transformation 
df['log_aq_total'] = np.log1p(df['aq_total'])
df['sqrt_age'] = np.sqrt(df['age'])

# interaction terms
df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
df['sqp_aq_interaction'] = df['spq_total'] * df['aq_total']
df['age_x_eq'] = df['age'] * df['eq_total']

# questionnaire score ratios 
df['aq_spq_ratio'] = df['aq_total'] / (df['spq_total'] + 1e-8)
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

#boolean: high aq (6 and above)
df['high_aq'] = (df['aq_total'] >= 6).astype(int)

# 2. feature reduction/selection

# remove highly correlated features 
# Only use numeric columns for correlation
numeric_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df = df.drop(columns=to_drop)

# drop low variance features 
# Only apply VarianceThreshold to numeric columns
feature_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
selector = VarianceThreshold(threshold=0.1)
selector.fit(df[feature_cols])
low_variance_cols = feature_cols[~selector.get_support()]
df = df.drop(columns=low_variance_cols)

# 3. one-hot encode new categorical features 
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)

# 4. save engineered dataset 
df.to_csv('../data/processed/data_c4_balanced_fe.csv', index=False)

print("feature engineering complete. new shape:", df.shape)
print("columns:", df.columns.tolist())

# baseline models

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# df load data
df = pd.read_csv('data/processed/data_c4_balanced_fe.csv')
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Scale features for Logistic Regression only
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x_imputed), columns=x_imputed.columns)

x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)
x_train_scaled, x_val_scaled, y_train_scaled, y_val_scaled = train_test_split(x_scaled, y, stratify=y, test_size=0.2, random_state=42)

# Store all models and their results
models = {}
results = {}

# 1. Logistic Regression
logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logreg.fit(x_train_scaled, y_train_scaled)
logreg_pred = logreg.predict(x_val_scaled)
logreg_proba = logreg.predict_proba(x_val_scaled)[:, 1]
logreg_auc = roc_auc_score(y_val_scaled, logreg_proba)
logreg_f1 = f1_score(y_val_scaled, logreg_pred)

models['logreg'] = logreg
results['logreg'] = {'auc': logreg_auc, 'f1': logreg_f1, 'model': logreg, 'data': 'scaled'}

print("Logistic Regression:")
print(f"ROC-AUC: {logreg_auc:.4f}")
print(f"F1-Score: {logreg_f1:.4f}")
print(classification_report(y_val_scaled, logreg_pred))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_val)
rf_proba = rf.predict_proba(x_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_proba)
rf_f1 = f1_score(y_val, rf_pred)

models['rf'] = rf
results['rf'] = {'auc': rf_auc, 'f1': rf_f1, 'model': rf, 'data': 'unscaled'}

print("\nRandom Forest:")
print(f"ROC-AUC: {rf_auc:.4f}")
print(f"F1-Score: {rf_f1:.4f}")
print(classification_report(y_val, rf_pred))

# 3. XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_val)
xgb_proba = xgb.predict_proba(x_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_proba)
xgb_f1 = f1_score(y_val, xgb_pred)

models['xgb'] = xgb
results['xgb'] = {'auc': xgb_auc, 'f1': xgb_f1, 'model': xgb, 'data': 'unscaled'}

print("\nXGBoost:")
print(f"ROC-AUC: {xgb_auc:.4f}")
print(f"F1-Score: {xgb_f1:.4f}")
print(classification_report(y_val, xgb_pred))

# 4. LightGBM
lgb = LGBMClassifier(random_state=42, verbose=-1)
lgb.fit(x_train, y_train)
lgb_pred = lgb.predict(x_val)
lgb_proba = lgb.predict_proba(x_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_proba)
lgb_f1 = f1_score(y_val, lgb_pred)

models['lgb'] = lgb
results['lgb'] = {'auc': lgb_auc, 'f1': lgb_f1, 'model': lgb, 'data': 'unscaled'}

print("\nLightGBM:")
print(f"ROC-AUC: {lgb_auc:.4f}")
print(f"F1-Score: {lgb_f1:.4f}")
print(classification_report(y_val, lgb_pred))

# 5. Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
gb_pred = gb.predict(x_val)
gb_proba = gb.predict_proba(x_val)[:, 1]
gb_auc = roc_auc_score(y_val, gb_proba)
gb_f1 = f1_score(y_val, gb_pred)

models['gb'] = gb
results['gb'] = {'auc': gb_auc, 'f1': gb_f1, 'model': gb, 'data': 'unscaled'}

print("\nGradient Boosting:")
print(f"ROC-AUC: {gb_auc:.4f}")
print(f"F1-Score: {gb_f1:.4f}")
print(classification_report(y_val, gb_pred))

# 6. Extra Trees (faster than RF, often better performance)
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(x_train, y_train)
et_pred = et.predict(x_val)
et_proba = et.predict_proba(x_val)[:, 1]
et_auc = roc_auc_score(y_val, et_proba)
et_f1 = f1_score(y_val, et_pred)

models['et'] = et
results['et'] = {'auc': et_auc, 'f1': et_f1, 'model': et, 'data': 'unscaled'}

print("\nExtra Trees:")
print(f"ROC-AUC: {et_auc:.4f}")
print(f"F1-Score: {et_f1:.4f}")
print(classification_report(y_val, et_pred))

# 7. AdaBoost (fast, often good performance)
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(x_train, y_train)
ada_pred = ada.predict(x_val)
ada_proba = ada.predict_proba(x_val)[:, 1]
ada_auc = roc_auc_score(y_val, ada_proba)
ada_f1 = f1_score(y_val, ada_pred)

models['ada'] = ada
results['ada'] = {'auc': ada_auc, 'f1': ada_f1, 'model': ada, 'data': 'unscaled'}

print("\nAdaBoost:")
print(f"ROC-AUC: {ada_auc:.4f}")
print(f"F1-Score: {ada_f1:.4f}")
print(classification_report(y_val, ada_pred))

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k]['auc'])
best_model = results[best_model_name]['model']
best_auc = results[best_model_name]['auc']
best_f1 = results[best_model_name]['f1']

print(f"\n{'='*50}")
print(f"BEST MODEL: {best_model_name.upper()}")
print(f"ROC-AUC: {best_auc:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"{'='*50}")

# Store best model for later use
best_model_name_final = best_model_name

# threshold tuning
- for best model only 

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report, roc_auc_score

print("\n" + "="*50)
print("THRESHOLD OPTIMIZATION FOR ALL MODELS")
print("="*50)

# Dictionary to store optimized thresholds and scores
optimized_results = {}

# Iterate through all models
for model_name, model_info in results.items():
    model = model_info['model']
    
    # Get predictions based on whether the model used scaled data or not
    if model_info['data'] == 'scaled':
        probs = model.predict_proba(x_val_scaled)[:, 1]
        x_val_current = x_val_scaled
        y_val_current = y_val
    else:
        probs = model.predict_proba(x_val)[:, 1]
        x_val_current = x_val
        y_val_current = y_val
    
    # Find best threshold for F1 score
    prec, rec, thresholds = precision_recall_curve(y_val_current, probs)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    
    # Handle case where thresholds might be empty
    if len(thresholds) > 0:
        best_thresh = thresholds[np.argmax(f1_scores[:-1])]  # Exclude the last item which doesn't have a threshold
    else:
        best_thresh = 0.5
    
    # Evaluate at best threshold
    pred_thresh = (probs >= best_thresh).astype(int)
    optimized_f1 = f1_score(y_val_current, pred_thresh)
    optimized_auc = roc_auc_score(y_val_current, probs)
    
    # Store results
    optimized_results[model_name] = {
        'threshold': best_thresh,
        'f1': optimized_f1,
        'auc': optimized_auc,
        'probs': probs,
        'x_val': x_val_current,
        'y_val': y_val_current
    }
    
    # Print results
    print(f"\n{model_name.upper()} - Best threshold for F1: {best_thresh:.3f}")
    print(f"F1 at best threshold: {optimized_f1:.4f}")
    print(f"ROC-AUC: {optimized_auc:.4f}")
    print(classification_report(y_val_current, pred_thresh))

# Find best model after threshold optimization
best_model_name_optimized = max(optimized_results.keys(), key=lambda k: optimized_results[k]['f1'])
best_thresh_optimized = optimized_results[best_model_name_optimized]['threshold']
best_f1_optimized = optimized_results[best_model_name_optimized]['f1']
best_auc_optimized = optimized_results[best_model_name_optimized]['auc']

print(f"\n{'='*50}")
print(f"BEST MODEL AFTER THRESHOLD OPTIMIZATION: {best_model_name_optimized.upper()}")
print(f"Best threshold: {best_thresh_optimized:.3f}")
print(f"F1-Score: {best_f1_optimized:.4f}")
print(f"ROC-AUC: {best_auc_optimized:.4f}")
print(f"{'='*50}")

# Store best model and data for feature importance
best_model_name_final = best_model_name_optimized
best_model_final = models[best_model_name_final]
best_x_val_final = optimized_results[best_model_name_final]['x_val']
best_y_val_final = optimized_results[best_model_name_final]['y_val']

# feature importance
- on best model only

In [ ]:
# Feature importance analysis for the BEST performing model only
import pandas as pd
from sklearn.inspection import permutation_importance

print(f"Feature Importance Analysis for {best_model_name_final.upper()} (Best Model)")
print("="*60)

# Get feature importance based on model type
if hasattr(best_model_final, 'feature_importances_'):
    # Tree-based models (RF, XGB, LGB, GB)
    importances = pd.Series(best_model_final.feature_importances_, index=x_train.columns)
    print(f"\nTop 20 features by {best_model_name_final.upper()} importance:")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
else:
    # Linear models (LogReg, SVM)
    if hasattr(best_model_final, 'coef_'):
        # Logistic Regression
        importances = pd.Series(np.abs(best_model_final.coef_[0]), index=x_train.columns)
    else:
        # SVM or other models
        importances = pd.Series(np.zeros(len(x_train.columns)), index=x_train.columns)
    
    print(f"\nTop 20 features by {best_model_name_final.upper()} coefficients (absolute values):")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Coefficient': importances
    }).sort_values('Coefficient', ascending=False)

# Permutation importance (more robust, works for all models)
print(f"\nComputing permutation importance for {best_model_name_final.upper()}...")
perm_importance = permutation_importance(
    best_model_final, 
    best_x_val_final, 
    best_y_val_final, 
    n_repeats=5, 
    random_state=42
)

perm_importance_df = pd.DataFrame({
    'Feature': x_train.columns,
    'Permutation_importance': perm_importance.importances_mean
}).sort_values('Permutation_importance', ascending=False)

print(f"\nTop 20 features by Permutation importance ({best_model_name_final.upper()}):")
print(perm_importance_df.head(20))

# Save both importance measures
print(f"\nSaving feature importance results for {best_model_name_final.upper()}...")
importance_df.to_csv(f'feature_importance_{best_model_name_final}.csv', index=False)
perm_importance_df.to_csv(f'permutation_importance_{best_model_name_final}.csv', index=False)

print(f"\nFeature importance analysis complete for {best_model_name_final.upper()}")
print(f"Results saved to: feature_importance_{best_model_name_final}.csv")
print(f"Permutation importance saved to: permutation_importance_{best_model_name_final}.csv")

# rerun models with individual items only 
- result = reduced performance

In [ ]:
# Remove total scores and domain aggregations
features_to_drop = ['aq_total', 'spq_total', 'eq_total', 'sqr_total', 'd_score']
x_individual = x.drop(columns=features_to_drop)

# Split the data again
x_train_ind, x_val_ind, y_train_ind, y_val_ind = train_test_split(
    x_individual, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with individual items
rf_ind = RandomForestClassifier(random_state=42)
rf_ind.fit(x_train_ind, y_train_ind)
rf_ind_probs = rf_ind.predict_proba(x_val_ind)[:, 1]

# Train XGBoost with individual items
xgb_ind = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_ind.fit(x_train_ind, y_train_ind)
xgb_ind_probs = xgb_ind.predict_proba(x_val_ind)[:, 1]

# Find best thresholds for F1 score
rf_ind_thresholds = np.linspace(0, 1, 100)
rf_ind_f1_scores = [f1_score(y_val_ind, (rf_ind_probs >= t).astype(int)) for t in rf_ind_thresholds]
rf_ind_best_thresh = rf_ind_thresholds[np.argmax(rf_ind_f1_scores)]

xgb_ind_thresholds = np.linspace(0, 1, 100)
xgb_ind_f1_scores = [f1_score(y_val_ind, (xgb_ind_probs >= t).astype(int)) for t in xgb_ind_thresholds]
xgb_ind_best_thresh = xgb_ind_thresholds[np.argmax(xgb_ind_f1_scores)]

# Evaluate models with individual items
print("\n--- Models with individual items only ---")
print(f"Random Forest - Best threshold for F1: {rf_ind_best_thresh:.3f}")
rf_ind_pred_thresh = (rf_ind_probs >= rf_ind_best_thresh).astype(int)
print("Random Forest validation set performance:")
print(classification_report(y_val_ind, rf_ind_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_ind, rf_ind_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_ind, rf_ind_probs):.3f}")

print(f"\nXGBoost - Best threshold for F1: {xgb_ind_best_thresh:.3f}")
xgb_ind_pred_thresh = (xgb_ind_probs >= xgb_ind_best_thresh).astype(int)
print("XGBoost validation set performance:")
print(classification_report(y_val_ind, xgb_ind_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_ind, xgb_ind_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_ind, xgb_ind_probs):.3f}")

# Keep only individual items and engineered features
# This will test if individual questions are more predictive than totals

# rerun model with only total scores 
- result = reduced performance

In [ ]:
# Keep only total scores and engineered features
individual_cols = [col for col in x.columns if any(col.startswith(prefix) for prefix in ['aq_', 'spq_', 'eq_', 'sqr_'])]
x_totals_only = x.drop(columns=individual_cols)

# Split the data
x_train_tot, x_val_tot, y_train_tot, y_val_tot = train_test_split(
    x_totals_only, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with total scores
rf_tot = RandomForestClassifier(random_state=42)
rf_tot.fit(x_train_tot, y_train_tot)
rf_tot_probs = rf_tot.predict_proba(x_val_tot)[:, 1]

# Train XGBoost with total scores
xgb_tot = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_tot.fit(x_train_tot, y_train_tot)
xgb_tot_probs = xgb_tot.predict_proba(x_val_tot)[:, 1]

# Find best thresholds for F1 score
rf_tot_thresholds = np.linspace(0, 1, 100)
rf_tot_f1_scores = [f1_score(y_val_tot, (rf_tot_probs >= t).astype(int)) for t in rf_tot_thresholds]
rf_tot_best_thresh = rf_tot_thresholds[np.argmax(rf_tot_f1_scores)]

xgb_tot_thresholds = np.linspace(0, 1, 100)
xgb_tot_f1_scores = [f1_score(y_val_tot, (xgb_tot_probs >= t).astype(int)) for t in xgb_tot_thresholds]
xgb_tot_best_thresh = xgb_tot_thresholds[np.argmax(xgb_tot_f1_scores)]

# Evaluate models with total scores
print("\n--- Models with total scores only ---")
print(f"Random Forest - Best threshold for F1: {rf_tot_best_thresh:.3f}")
rf_tot_pred_thresh = (rf_tot_probs >= rf_tot_best_thresh).astype(int)
print("Random Forest validation set performance:")
print(classification_report(y_val_tot, rf_tot_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_tot, rf_tot_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_tot, rf_tot_probs):.3f}")

print(f"\nXGBoost - Best threshold for F1: {xgb_tot_best_thresh:.3f}")
xgb_tot_pred_thresh = (xgb_tot_probs >= xgb_tot_best_thresh).astype(int)
print("XGBoost validation set performance:")
print(classification_report(y_val_tot, xgb_tot_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_tot, xgb_tot_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_tot, xgb_tot_probs):.3f}")

# This will test if aggregated scores are more predictive

# what features are being used by the model at this stage. 

In [ ]:
# Copy and paste this code into your notebook to see all features being used by the model

print("="*60)
print("ALL FEATURES USED BY THE MODEL")
print("="*60)
print(f"Total number of features: {len(x.columns)}")
print()

# Group features by category
demographic_features = ['age', 'sex_2.0', 'sex_3.0', 'sex_4.0', 'sex_unknown', 'is_stem_occupation']
total_scores = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                      'age_x_eq', 'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq',
                      'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+']

# Individual questionnaire items
spq_items = [col for col in x.columns if col.startswith('spq_') and col != 'spq_total']
eq_items = [col for col in x.columns if col.startswith('eq_') and col != 'eq_total']
sqr_items = [col for col in x.columns if col.startswith('sqr_') and col != 'sqr_total']
aq_items = [col for col in x.columns if col.startswith('aq_') and col != 'aq_total']

print("1. DEMOGRAPHIC FEATURES:")
for feat in demographic_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("2. TOTAL SCORES:")
for feat in total_scores:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("3. INDIVIDUAL QUESTIONNAIRE ITEMS:")
print("   SPQ items (10):", ", ".join(spq_items))
print("   EQ items (10):", ", ".join(eq_items))
print("   SQR items (10):", ", ".join(sqr_items))
print("   AQ items (10):", ", ".join(aq_items))
print()

print("4. ENGINEERED FEATURES:")
for feat in engineered_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()


print("="*60)
print("FEATURE SUMMARY")
print("="*60)
print(f"Demographic features: {len([f for f in demographic_features if f in x.columns])}")
print(f"Total scores: {len([f for f in total_scores if f in x.columns])}")
print(f"Individual SPQ items: {len(spq_items)}")
print(f"Individual EQ items: {len(eq_items)}")
print(f"Individual SQR items: {len(sqr_items)}")
print(f"Individual AQ items: {len(aq_items)}")
print(f"Engineered features: {len([f for f in engineered_features if f in x.columns])}")
print(f"TOTAL: {len(x.columns)} features")

# Suprised sex is not a bigger predictor in the feature importance list
- exploring potential explinations

In [ ]:
# Suprised sex is not a bigger predictor in the feature importance list
# exploring potential explinations

print("="*50)
print("INVESTIGATING SEX AS A PREDICTOR")
print("="*50)

# 1. Check sex distribution
print("1. SEX DISTRIBUTION:")
print("sex_2.0 (likely male):", df['sex_2.0'].sum())
print("sex_3.0 (likely female):", df['sex_3.0'].sum()) 
print("sex_4.0 (likely other):", df['sex_4.0'].sum())
print("sex_unknown:", df['sex_unknown'].sum())
print()

# 2. Check correlation between sex and target
print("2. CORRELATION WITH TARGET:")
print("sex_2.0 correlation with autism_target:", df['sex_2.0'].corr(df['autism_target']))
print("sex_3.0 correlation with autism_target:", df['sex_3.0'].corr(df['autism_target']))
print("sex_4.0 correlation with autism_target:", df['sex_4.0'].corr(df['autism_target']))
print("sex_unknown correlation with autism_target:", df['sex_unknown'].corr(df['autism_target']))
print()

# 3. Check AQ scores by sex
print("3. AQ SCORES BY SEX:")
if 'sex_2.0' in df.columns and df['sex_2.0'].sum() > 0:
    male_aq = df[df['sex_2.0'] == 1]['aq_total'].mean()
    print(f"Average AQ score for sex_2.0: {male_aq:.2f}")
if 'sex_3.0' in df.columns and df['sex_3.0'].sum() > 0:
    female_aq = df[df['sex_3.0'] == 1]['aq_total'].mean()
    print(f"Average AQ score for sex_3.0: {female_aq:.2f}")
print()

# 4. Check autism prevalence by sex
print("4. AUTISM PREVALENCE BY SEX:")
if 'sex_2.0' in df.columns and df['sex_2.0'].sum() > 0:
    male_autism_rate = df[df['sex_2.0'] == 1]['autism_target'].mean()
    print(f"Autism rate for sex_2.0: {male_autism_rate:.3f} ({male_autism_rate*100:.1f}%)")
if 'sex_3.0' in df.columns and df['sex_3.0'].sum() > 0:
    female_autism_rate = df[df['sex_3.0'] == 1]['autism_target'].mean()
    print(f"Autism rate for sex_3.0: {female_autism_rate:.3f} ({female_autism_rate*100:.1f}%)")
print()

# 5. Check feature importance for sex variables
print("5. SEX FEATURE IMPORTANCE (from best model):")
sex_features = ['sex_2.0', 'sex_3.0', 'sex_4.0', 'sex_unknown']
for feat in sex_features:
    if feat in x.columns:
        # Get importance from the best model (assuming it's stored)
        if hasattr(best_model_final, 'feature_importances_'):
            feat_idx = list(x.columns).index(feat)
            importance = best_model_final.feature_importances_[feat_idx]
            print(f"{feat}: {importance:.6f}")
        else:
            print(f"{feat}: importance not available")
print()

print("="*50)
print("POTENTIAL EXPLANATIONS:")
print("1. Sex might be captured through other features (age, questionnaire patterns)")
print("2. Dataset might be balanced by sex, reducing its predictive power")
print("3. Sex differences might be subtle and captured by interaction effects")
print("4. One-hot encoding might create sparse features that are hard to learn")
print("="*50)

# this revealed a huge imbalance
- dataset was >95% male 
- below us investigating why by comparing raw datset

In [ ]:
# Copy and paste this code to investigate the sex distribution issue

import pandas as pd
import numpy as np

print("="*60)
print("INVESTIGATING SEX DISTRIBUTION ISSUE")
print("="*60)

# 1. Check original raw dataset
print("1. ORIGINAL RAW DATASET:")
print("Loading original dataset...")
try:
    df_raw = pd.read_csv('/Users/eb2007/documents/phd/data/data_c4_raw.csv')
    print(f"Original dataset shape: {df_raw.shape}")
    
    # Check what sex columns exist in raw data
    sex_cols_raw = [col for col in df_raw.columns if 'sex' in col.lower()]
    print(f"Sex-related columns in raw data: {sex_cols_raw}")
    
    # Show unique values in sex columns
    for col in sex_cols_raw:
        if col in df_raw.columns:
            print(f"\n{col} unique values:")
            print(df_raw[col].value_counts())
            print(f"Missing values: {df_raw[col].isnull().sum()}")
    
except Exception as e:
    print(f"Error loading raw dataset: {e}")

print("\n" + "="*60)

# 2. Check processed dataset
print("2. PROCESSED DATASET:")
print("Loading processed dataset...")
try:
    df_processed = pd.read_csv('data/processed/data_c4_matched_balanced.csv')
    print(f"Processed dataset shape: {df_processed.shape}")
    
    # Check what sex columns exist in processed data
    sex_cols_processed = [col for col in df_processed.columns if 'sex' in col.lower()]
    print(f"Sex-related columns in processed data: {sex_cols_processed}")
    
    # Show unique values in sex columns
    for col in sex_cols_processed:
        if col in df_processed.columns:
            print(f"\n{col} unique values:")
            print(df_processed[col].value_counts())
            print(f"Missing values: {df_processed[col].isnull().sum()}")
    
except Exception as e:
    print(f"Error loading processed dataset: {e}")

print("\n" + "="*60)

# 3. Compare sex distributions
print("3. COMPARISON:")
if 'df_raw' in locals() and 'df_processed' in locals():
    print("Sex distribution comparison:")
    
    # Find common sex columns or try to identify them
    raw_sex_col = None
    processed_sex_cols = []
    
    # Look for sex columns in raw data
    for col in df_raw.columns:
        if 'sex' in col.lower():
            raw_sex_col = col
            break
    
    # Look for sex columns in processed data
    for col in df_processed.columns:
        if 'sex' in col.lower():
            processed_sex_cols.append(col)
    
    if raw_sex_col:
        print(f"\nRaw dataset - {raw_sex_col}:")
        print(df_raw[raw_sex_col].value_counts())
        print(f"Total: {len(df_raw)}")
    
    if processed_sex_cols:
        print(f"\nProcessed dataset - sex columns:")
        for col in processed_sex_cols:
            print(f"{col}: {df_processed[col].sum()} ({df_processed[col].sum()/len(df_processed)*100:.1f}%)")
        print(f"Total: {len(df_processed)}")


# re-examine OG dataset 

In [ ]:
# Check the original sex distribution by autism status
df_raw = pd.read_csv('/Users/eb2007/documents/phd/data/data_c4_raw.csv')

# First, create the autism_target column using the same logic from the notebooks
print("Creating autism_target column...")

# Get diagnosis columns
diagnosis_cols = [col for col in df_raw.columns if col.startswith('diagnosis_') and not col.startswith('autism_diagnosis')]
autism_diagnosis_cols = [col for col in df_raw.columns if col.startswith('autism_diagnosis')]

print(f"Diagnosis columns: {diagnosis_cols}")
print(f"Autism diagnosis columns: {autism_diagnosis_cols}")

# Convert to numeric
for col in diagnosis_cols + autism_diagnosis_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Create autism target using the same logic as in the notebooks
# Method 1: Look for value 2 in diagnosis columns
autism_mask_1 = df_raw[diagnosis_cols] == 2
autism_from_diagnosis = autism_mask_1.any(axis=1)

# Method 2: Look for values 1,2,3 in autism_diagnosis columns
if autism_diagnosis_cols:
    autism_diagnosis_mask = ((df_raw[autism_diagnosis_cols] == 1) | 
                            (df_raw[autism_diagnosis_cols] == 2) | 
                            (df_raw[autism_diagnosis_cols] == 3)).any(axis=1)
    df_raw['autism_target'] = (autism_from_diagnosis | autism_diagnosis_mask).astype(int)
else:
    df_raw['autism_target'] = autism_from_diagnosis.astype(int)

print(f"Autism target distribution: {df_raw['autism_target'].value_counts().to_dict()}")

# Now check sex distribution for autism cases vs controls
autism_cases = df_raw[df_raw['autism_target'] == 1]
control_cases = df_raw[df_raw['autism_target'] == 0]

print("\nAutism cases by sex:")
print(autism_cases['sex'].value_counts())
print(f"Total autism cases: {len(autism_cases)}")

print("\nControl cases by sex:")
print(control_cases['sex'].value_counts())
print(f"Total control cases: {len(control_cases)}")

# Calculate percentages
print("\nSex distribution percentages:")
print("Autism cases:")
for sex_val, count in autism_cases['sex'].value_counts().items():
    percentage = (count / len(autism_cases)) * 100
    print(f"  Sex {sex_val}: {count} ({percentage:.1f}%)")

print("\nControl cases:")
for sex_val, count in control_cases['sex'].value_counts().items():
    percentage = (count / len(control_cases)) * 100
    print(f"  Sex {sex_val}: {count} ({percentage:.1f}%)")

# re-do matching with sex stratification 


In [ ]:
# Create a properly sex-balanced dataset
def create_sex_balanced_dataset(df):
    balanced_data = []
    
    for sex_value in [1.0, 2.0, 3.0, 4.0]:  # Process each sex separately
        sex_data = df[df['sex'] == sex_value]
        autism_cases = sex_data[sex_data['autism_target'] == 1]
        control_cases = sex_data[sex_data['autism_target'] == 0]
        
        print(f"Sex {sex_value}: {len(autism_cases)} autism, {len(control_cases)} controls")
        
        # Sample equal numbers for this sex
        min_count = min(len(autism_cases), len(control_cases))
        if min_count > 0:
            autism_balanced = autism_cases.sample(n=min_count, random_state=42)
            control_balanced = control_cases.sample(n=min_count, random_state=42)
            balanced_data.append(pd.concat([autism_balanced, control_balanced]))
    
    return pd.concat(balanced_data, ignore_index=True)

# Create the balanced dataset
df_balanced = create_sex_balanced_dataset(df_raw)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print(f"Balanced autism target distribution: {df_balanced['autism_target'].value_counts().to_dict()}")
print(f"Balanced sex distribution: {df_balanced['sex'].value_counts().to_dict()}")

# Save the properly balanced dataset
df_balanced.to_csv('data/processed/data_c4_properly_balanced.csv', index=False)

# re run sex investigation
- added new feature engineering cell below to use: df = pd.read_csv('data/processed/data_c4_properly_balanced.csv')

In [ ]:
# Re-run your sex investigation with the new balanced dataset
print("="*60)
print("SEX DISTRIBUTION IN NEW BALANCED DATASET")
print("="*60)

# Load the new engineered dataset
df = pd.read_csv('data/processed/data_c4_balanced_fe.csv')  # Updated engineered dataset

# Check sex distribution
sex_features = ['sex_2.0', 'sex_3.0', 'sex_4.0', 'sex_unknown']
for feat in sex_features:
    if feat in df.columns:
        print(f"{feat}: {df[feat].sum()} ({df[feat].sum()/len(df)*100:.1f}%)")

# Check correlation with target
for feat in sex_features:
    if feat in df.columns:
        corr = df[feat].corr(df['autism_target'])
        print(f"{feat} correlation with autism_target: {corr:.4f}")

# new feature engineering for properly balanced dataset (Cell 28)

In [ ]:
# CORRECTED FEATURE ENGINEERING - MATCHING ORIGINAL FEATURE SET
print("="*60)
print("CORRECTED FEATURE ENGINEERING - MATCHING ORIGINAL FEATURES")
print("="*60)

import pandas as pd
import numpy as np 
from sklearn.feature_selection import VarianceThreshold

# Load the NEW properly balanced dataset
df = pd.read_csv('data/processed/data_c4_properly_balanced.csv')
print(f"Loaded properly balanced dataset: {df.shape}")

# REMOVE userid - it should never be a feature
if 'userid' in df.columns:
    df = df.drop(columns=['userid'])
    print("Removed userid column")

# Create missing total scores
print("\nCreating missing total scores...")

# Create AQ total if it doesn't exist
if 'aq_total' not in df.columns:
    aq_cols = [col for col in df.columns if col.startswith('aq_') and col != 'aq_total']
    if aq_cols:
        df['aq_total'] = df[aq_cols].sum(axis=1)
        print(f"Created aq_total from {len(aq_cols)} AQ items")

# Create SPQ total if it doesn't exist
if 'spq_total' not in df.columns:
    spq_cols = [col for col in df.columns if col.startswith('spq_') and col != 'spq_total']
    if spq_cols:
        df['spq_total'] = df[spq_cols].sum(axis=1)
        print(f"Created spq_total from {len(spq_cols)} SPQ items")

# Create EQ total if it doesn't exist
if 'eq_total' not in df.columns:
    eq_cols = [col for col in df.columns if col.startswith('eq_') and col != 'eq_total']
    if eq_cols:
        df['eq_total'] = df[eq_cols].sum(axis=1)
        print(f"Created eq_total from {len(eq_cols)} EQ items")

# Create SQR total if it doesn't exist
if 'sqr_total' not in df.columns:
    sqr_cols = [col for col in df.columns if col.startswith('sqr_') and col != 'sqr_total']
    if sqr_cols:
        df['sqr_total'] = df[sqr_cols].sum(axis=1)
        print(f"Created sqr_total from {len(sqr_cols)} SQR items")

# Create d_score if it doesn't exist
if 'd_score' not in df.columns and 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['d_score'] = df['eq_total'] - df['sqr_total']
    print("Created d_score")

print(f"\nAfter creating totals, shape: {df.shape}")

# 1. feature creation - EXACTLY AS ORIGINAL
print("\nCreating engineered features...")

# age group bins
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], labels=['0-18', '19-30', '31-45', '46-60', '61+'])

# non linear transformation 
df['log_aq_total'] = np.log1p(df['aq_total'])
df['sqrt_age'] = np.sqrt(df['age'])

# interaction terms
df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
df['sqp_aq_interaction'] = df['spq_total'] * df['aq_total']
df['age_x_eq'] = df['age'] * df['eq_total']
df['age_x_aq'] = df['age'] * df['aq_total']

# questionnaire score ratios 
df['aq_spq_ratio'] = df['aq_total'] / (df['spq_total'] + 1e-8)
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

# boolean: high aq (6 and above) - UPDATED THRESHOLD
df['high_aq'] = (df['aq_total'] >= 6).astype(int)

# STEM occupation - FIXED to handle non-string data
if 'occupation' in df.columns:
    # Convert to string first, then check for STEM keywords
    df['is_stem_occupation'] = df['occupation'].astype(str).str.contains(
        'stem|science|technology|engineering|math', case=False, na=False
    ).astype(int)
    print("Created is_stem_occupation feature")
else:
    # If occupation column doesn't exist, create a dummy column
    df['is_stem_occupation'] = 0
    print("Occupation column not found, created dummy is_stem_occupation")

# 2. CORRECTED: One-hot encode sex WITHOUT dropping first category
print("\nOne-hot encoding sex (CORRECTED)...")
df = pd.get_dummies(df, columns=['sex'], prefix='sex', drop_first=False)

# 3. One-hot encode age groups
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)

print(f"After encoding, shape: {df.shape}")

# 4. REMOVE DATA LEAKAGE - Remove diagnosis columns
print("\nRemoving data leakage columns...")
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"Removing diagnosis columns: {diagnosis_cols}")
df = df.drop(columns=diagnosis_cols)
print(f"After removing diagnosis columns, shape: {df.shape}")

# 5. feature reduction/selection - LESS AGGRESSIVE
print("\nApplying feature selection...")

# remove highly correlated features (less aggressive threshold)
numeric_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.98)]  # Increased from 0.95
df = df.drop(columns=to_drop)

# drop low variance features (less aggressive threshold)
feature_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
selector = VarianceThreshold(threshold=0.05)  # Reduced from 0.1
selector.fit(df[feature_cols])
low_variance_cols = feature_cols[~selector.get_support()]
df = df.drop(columns=low_variance_cols)

# 6. save engineered dataset 
df.to_csv('data/processed/data_c4_properly_balanced_fe.csv', index=False)

print(f"\nFeature engineering complete. Final shape: {df.shape}")

# Check sex distribution in the new engineered dataset
print("\n" + "="*60)
print("SEX DISTRIBUTION IN NEW ENGINEERED DATASET")
print("="*60)

sex_features = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']
for feat in sex_features:
    if feat in df.columns:
        print(f"{feat}: {df[feat].sum()} ({df[feat].sum()/len(df)*100:.1f}%)")
        corr = df[feat].corr(df['autism_target'])
        print(f"  Correlation with autism_target: {corr:.4f}")

print(f"\nTotal samples: {len(df)}")
print(f"Autism target distribution: {df['autism_target'].value_counts().to_dict()}")

# Show final feature list
print("\n" + "="*60)
print("FINAL FEATURE LIST")
print("="*60)
feature_list = [col for col in df.columns if col != 'autism_target']
print(f"Total features: {len(feature_list)}")
print("Features:", feature_list)

# re-run baseline models with properly balanced dataset 

In [ ]:
# BASELINE MODELS WITH PROPERLY BALANCED DATASET (CLEAN)
print("="*60)
print("BASELINE MODELS WITH PROPERLY BALANCED DATASET (CLEAN)")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load the properly balanced engineered dataset (should be clean now)
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
print(f"Loaded properly balanced engineered dataset: {df.shape}")

# Check for any remaining diagnosis columns (should be none)
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
if diagnosis_cols:
    print(f"WARNING: Diagnosis columns still present: {diagnosis_cols}")
    print("Removing them now...")
    df = df.drop(columns=diagnosis_cols)
    print(f"After removal, shape: {df.shape}")
else:
    print("✓ No diagnosis columns found - dataset is clean")

# Prepare features and target
x = df.drop(columns=['autism_target'])
y = df['autism_target']

print(f"Features shape: {x.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle missing values
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Scale features for Logistic Regression only
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x_imputed), columns=x_imputed.columns)

# Split data
x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)
x_train_scaled, x_val_scaled, y_train_scaled, y_val_scaled = train_test_split(x_scaled, y, stratify=y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")

# Store all models and their results
models = {}
results = {}

# 1. Logistic Regression
print("\n1. Training Logistic Regression...")
logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logreg.fit(x_train_scaled, y_train_scaled)
logreg_pred = logreg.predict(x_val_scaled)
logreg_proba = logreg.predict_proba(x_val_scaled)[:, 1]
logreg_auc = roc_auc_score(y_val_scaled, logreg_proba)
logreg_f1 = f1_score(y_val_scaled, logreg_pred)

models['logreg'] = logreg
results['logreg'] = {'auc': logreg_auc, 'f1': logreg_f1, 'model': logreg, 'data': 'scaled'}

print(f"Logistic Regression - ROC-AUC: {logreg_auc:.4f}, F1: {logreg_f1:.4f}")

# 2. Random Forest
print("\n2. Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_val)
rf_proba = rf.predict_proba(x_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_proba)
rf_f1 = f1_score(y_val, rf_pred)

models['rf'] = rf
results['rf'] = {'auc': rf_auc, 'f1': rf_f1, 'model': rf, 'data': 'unscaled'}

print(f"Random Forest - ROC-AUC: {rf_auc:.4f}, F1: {rf_f1:.4f}")

# 3. XGBoost
print("\n3. Training XGBoost...")
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_val)
xgb_proba = xgb.predict_proba(x_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_proba)
xgb_f1 = f1_score(y_val, xgb_pred)

models['xgb'] = xgb
results['xgb'] = {'auc': xgb_auc, 'f1': xgb_f1, 'model': xgb, 'data': 'unscaled'}

print(f"XGBoost - ROC-AUC: {xgb_auc:.4f}, F1: {xgb_f1:.4f}")

# 4. LightGBM
print("\n4. Training LightGBM...")
lgb = LGBMClassifier(random_state=42, verbose=-1)
lgb.fit(x_train, y_train)
lgb_pred = lgb.predict(x_val)
lgb_proba = lgb.predict_proba(x_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_proba)
lgb_f1 = f1_score(y_val, lgb_pred)

models['lgb'] = lgb
results['lgb'] = {'auc': lgb_auc, 'f1': lgb_f1, 'model': lgb, 'data': 'unscaled'}

print(f"LightGBM - ROC-AUC: {lgb_auc:.4f}, F1: {lgb_f1:.4f}")

# 5. Gradient Boosting
print("\n5. Training Gradient Boosting...")
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
gb_pred = gb.predict(x_val)
gb_proba = gb.predict_proba(x_val)[:, 1]
gb_auc = roc_auc_score(y_val, gb_proba)
gb_f1 = f1_score(y_val, gb_pred)

models['gb'] = gb
results['gb'] = {'auc': gb_auc, 'f1': gb_f1, 'model': gb, 'data': 'unscaled'}

print(f"Gradient Boosting - ROC-AUC: {gb_auc:.4f}, F1: {gb_f1:.4f}")

# 6. Extra Trees
print("\n6. Training Extra Trees...")
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(x_train, y_train)
et_pred = et.predict(x_val)
et_proba = et.predict_proba(x_val)[:, 1]
et_auc = roc_auc_score(y_val, et_proba)
et_f1 = f1_score(y_val, et_pred)

models['et'] = et
results['et'] = {'auc': et_auc, 'f1': et_f1, 'model': et, 'data': 'unscaled'}

print(f"Extra Trees - ROC-AUC: {et_auc:.4f}, F1: {et_f1:.4f}")

# 7. AdaBoost
print("\n7. Training AdaBoost...")
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(x_train, y_train)
ada_pred = ada.predict(x_val)
ada_proba = ada.predict_proba(x_val)[:, 1]
ada_auc = roc_auc_score(y_val, ada_proba)
ada_f1 = f1_score(y_val, ada_pred)

models['ada'] = ada
results['ada'] = {'auc': ada_auc, 'f1': ada_f1, 'model': ada, 'data': 'unscaled'}

print(f"AdaBoost - ROC-AUC: {ada_auc:.4f}, F1: {ada_f1:.4f}")

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k]['auc'])
best_model = results[best_model_name]['model']
best_auc = results[best_model_name]['auc']
best_f1 = results[best_model_name]['f1']

print(f"\n{'='*50}")
print(f"BEST MODEL: {best_model_name.upper()}")
print(f"ROC-AUC: {best_auc:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"{'='*50}")

# Store best model for later use
best_model_name_final = best_model_name

# Show all results in a summary table
print(f"\n{'='*60}")
print("SUMMARY OF ALL MODEL PERFORMANCES")
print("="*60)
print(f"{'Model':<15} {'ROC-AUC':<10} {'F1-Score':<10}")
print("-" * 35)
for model_name, result in results.items():
    print(f"{model_name.upper():<15} {result['auc']:<10.4f} {result['f1']:<10.4f}")
print("="*60)

In [ ]:
# QUICK CROSS-VALIDATION CHECK
print("="*50)
print("CROSS-VALIDATION CHECK")
print("="*50)

from sklearn.model_selection import cross_val_score

# Test the best model with cross-validation
best_model = results[best_model_name_final]['model']
cv_scores = cross_val_score(best_model, x_imputed, y, cv=5, scoring='roc_auc')

print(f"Cross-validation ROC-AUC scores: {cv_scores}")
print(f"Mean CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Check if CV score is similar to validation score
val_score = results[best_model_name_final]['auc']
print(f"Validation score: {val_score:.4f}")
print(f"Difference: {abs(val_score - cv_scores.mean()):.4f}")

if abs(val_score - cv_scores.mean()) < 0.05:
    print("✓ Cross-validation confirms results are stable")
else:
    print("⚠️ Large difference between CV and validation scores")

# threshold optimization

In [ ]:
# THRESHOLD OPTIMIZATION FOR PROPERLY BALANCED DATASET
print("="*60)
print("THRESHOLD OPTIMIZATION FOR PROPERLY BALANCED DATASET")
print("="*60)

from sklearn.metrics import precision_recall_curve, f1_score, classification_report, roc_auc_score

# Dictionary to store optimized thresholds and scores
optimized_results = {}

# Iterate through all models
for model_name, model_info in results.items():
    model = model_info['model']
    
    # Get predictions based on whether the model used scaled data or not
    if model_info['data'] == 'scaled':
        probs = model.predict_proba(x_val_scaled)[:, 1]
        x_val_current = x_val_scaled
        y_val_current = y_val
    else:
        probs = model.predict_proba(x_val)[:, 1]
        x_val_current = x_val
        y_val_current = y_val
    
    # Find best threshold for F1 score
    prec, rec, thresholds = precision_recall_curve(y_val_current, probs)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    
    # Handle case where thresholds might be empty
    if len(thresholds) > 0:
        best_thresh = thresholds[np.argmax(f1_scores[:-1])]
    else:
        best_thresh = 0.5
    
    # Evaluate at best threshold
    pred_thresh = (probs >= best_thresh).astype(int)
    optimized_f1 = f1_score(y_val_current, pred_thresh)
    optimized_auc = roc_auc_score(y_val_current, probs)
    
    # Store results
    optimized_results[model_name] = {
        'threshold': best_thresh,
        'f1': optimized_f1,
        'auc': optimized_auc,
        'probs': probs,
        'x_val': x_val_current,
        'y_val': y_val_current
    }
    
    # Print results
    print(f"\n{model_name.upper()} - Best threshold for F1: {best_thresh:.3f}")
    print(f"F1 at best threshold: {optimized_f1:.4f}")
    print(f"ROC-AUC: {optimized_auc:.4f}")
    print(classification_report(y_val_current, pred_thresh))

# Find best model after threshold optimization
best_model_name_optimized = max(optimized_results.keys(), key=lambda k: optimized_results[k]['f1'])
best_thresh_optimized = optimized_results[best_model_name_optimized]['threshold']
best_f1_optimized = optimized_results[best_model_name_optimized]['f1']
best_auc_optimized = optimized_results[best_model_name_optimized]['auc']

print(f"\n{'='*50}")
print(f"BEST MODEL AFTER THRESHOLD OPTIMIZATION: {best_model_name_optimized.upper()}")
print(f"Best threshold: {best_thresh_optimized:.3f}")
print(f"F1-Score: {best_f1_optimized:.4f}")
print(f"ROC-AUC: {best_auc_optimized:.4f}")
print(f"{'='*50}")

# Store best model and data for feature importance
best_model_name_final = best_model_name_optimized
best_model_final = models[best_model_name_final]
best_x_val_final = optimized_results[best_model_name_final]['x_val']
best_y_val_final = optimized_results[best_model_name_final]['y_val']

# Show all results in a summary table
print(f"\n{'='*60}")
print("SUMMARY OF ALL MODEL PERFORMANCES AFTER THRESHOLD OPTIMIZATION")
print("="*60)
print(f"{'Model':<15} {'Threshold':<12} {'F1-Score':<10} {'ROC-AUC':<10}")
print("-" * 47)
for model_name, result in optimized_results.items():
    print(f"{model_name.upper():<15} {result['threshold']:<12.3f} {result['f1']:<10.4f} {result['auc']:<10.4f}")
print("="*60)

# feature importance 

In [ ]:
# FEATURE IMPORTANCE FOR PROPERLY BALANCED DATASET
print("="*60)
print("FEATURE IMPORTANCE FOR PROPERLY BALANCED DATASET")
print("="*60)

import pandas as pd
from sklearn.inspection import permutation_importance

print(f"Feature Importance Analysis for {best_model_name_final.upper()} (Best Model)")
print("="*60)

# Get feature importance based on model type
if hasattr(best_model_final, 'feature_importances_'):
    # Tree-based models (RF, XGB, LGB, GB)
    importances = pd.Series(best_model_final.feature_importances_, index=x_train.columns)
    print(f"\nTop 20 features by {best_model_name_final.upper()} importance:")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
else:
    # Linear models (LogReg, SVM)
    if hasattr(best_model_final, 'coef_'):
        # Logistic Regression
        importances = pd.Series(np.abs(best_model_final.coef_[0]), index=x_train.columns)
    else:
        # SVM or other models
        importances = pd.Series(np.zeros(len(x_train.columns)), index=x_train.columns)
    
    print(f"\nTop 20 features by {best_model_name_final.upper()} coefficients (absolute values):")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Coefficient': importances
    }).sort_values('Coefficient', ascending=False)

# Permutation importance (more robust, works for all models)
print(f"\nComputing permutation importance for {best_model_name_final.upper()}...")
perm_importance = permutation_importance(
    best_model_final, 
    best_x_val_final, 
    best_y_val_final, 
    n_repeats=5, 
    random_state=42
)

perm_importance_df = pd.DataFrame({
    'Feature': x_train.columns,
    'Permutation_importance': perm_importance.importances_mean
}).sort_values('Permutation_importance', ascending=False)

print(f"\nTop 20 features by Permutation importance ({best_model_name_final.upper()}):")
print(perm_importance_df.head(20))

# Save both importance measures
print(f"\nSaving feature importance results for {best_model_name_final.upper()}...")
importance_df.to_csv(f'feature_importance_{best_model_name_final}_balanced.csv', index=False)
perm_importance_df.to_csv(f'permutation_importance_{best_model_name_final}_balanced.csv', index=False)

print(f"\nFeature importance analysis complete for {best_model_name_final.upper()}")
print(f"Results saved to: feature_importance_{best_model_name_final}_balanced.csv")
print(f"Permutation importance saved to: permutation_importance_{best_model_name_final}_balanced.csv")

# Check sex feature importance specifically
print(f"\n{'='*50}")
print("SEX FEATURE IMPORTANCE ANALYSIS")
print("="*50)

sex_features = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']
for feat in sex_features:
    if feat in x_train.columns:
        if hasattr(best_model_final, 'feature_importances_'):
            importance = importances[feat]
            print(f"{feat}: {importance:.6f}")
        else:
            coef = importances[feat]
            print(f"{feat}: {coef:.6f}")
        
        # Check permutation importance
        perm_imp = perm_importance_df[perm_importance_df['Feature'] == feat]['Permutation_importance'].values
        if len(perm_imp) > 0:
            print(f"  Permutation importance: {perm_imp[0]:.6f}")

# updating features and aligning with correct model
- removing problematic features 
- then checking features are correct 

In [ ]:
# CLEAN DATASET - REMOVE PROBLEMATIC FEATURES
print("="*60)
print("CLEANING DATASET - REMOVING PROBLEMATIC FEATURES")
print("="*60)

# Load the current dataset
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
print(f"Original shape: {df.shape}")

# Remove problematic columns
columns_to_remove = ['repeat', 'handedness', 'country_region', 'education', 'occupation']
for col in columns_to_remove:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"Removed {col} column")

print(f"After cleaning, shape: {df.shape}")

# Add missing engineered features
print("\nAdding missing engineered features...")

# Add log_aq_total if missing
if 'log_aq_total' not in df.columns and 'aq_total' in df.columns:
    df['log_aq_total'] = np.log1p(df['aq_total'])
    print("Added log_aq_total")

# Add sqrt_age if missing
if 'sqrt_age' not in df.columns and 'age' in df.columns:
    df['sqrt_age'] = np.sqrt(df['age'])
    print("Added sqrt_age")

# Add eq_sqr_ratio if missing
if 'eq_sqr_ratio' not in df.columns and 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    print("Added eq_sqr_ratio")

# Add high_aq if missing
if 'high_aq' not in df.columns and 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] >= 6).astype(int)
    print("Added high_aq")

# Add is_stem_occupation if missing (set to 0 since occupation was removed)
if 'is_stem_occupation' not in df.columns:
    df['is_stem_occupation'] = 0
    print("Added is_stem_occupation (dummy)")

print(f"Final shape: {df.shape}")

# Save the cleaned dataset
df.to_csv('data/processed/data_c4_properly_balanced_fe.csv', index=False)
print("Saved cleaned dataset")

# Verify the cleaning
print("\n" + "="*60)
print("VERIFICATION - CLEANED DATASET")
print("="*60)

# Check problematic features are gone
problematic_features = ['userid', 'repeat', 'handedness', 'country_region', 'education', 'occupation']
for feat in problematic_features:
    if feat in df.columns:
        print(f"WARNING: {feat} - STILL PRESENT")
    else:
        print(f"OK: {feat} - REMOVED")

# Check sex features are still there
sex_features = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']
for feat in sex_features:
    if feat in df.columns:
        print(f"OK: {feat} - PRESENT")
    else:
        print(f"ERROR: {feat} - MISSING")

# Check engineered features
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                      'age_x_eq', 'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq',
                      'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+',
                      'is_stem_occupation']

print(f"\nEngineered features present: {len([f for f in engineered_features if f in df.columns])}/{len(engineered_features)}")

# Final feature count
x_clean = df.drop(columns=['autism_target'])
print(f"\nFinal feature count: {len(x_clean.columns)}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")

In [ ]:
# CHECK FEATURES IN NEW DATASET
print("="*60)
print("ALL FEATURES USED BY THE MODEL (NEW DATASET)")
print("="*60)

# Load the dataset
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
x = df.drop(columns=['autism_target'])

print(f"Total number of features: {len(x.columns)}")
print()

# Group features by category
demographic_features = ['age', 'sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0', 'is_stem_occupation']
total_scores = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                      'age_x_eq', 'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq',
                      'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+']

# Individual questionnaire items
spq_items = [col for col in x.columns if col.startswith('spq_') and col != 'spq_total']
eq_items = [col for col in x.columns if col.startswith('eq_') and col != 'eq_total']
sqr_items = [col for col in x.columns if col.startswith('sqr_') and col != 'sqr_total']
aq_items = [col for col in x.columns if col.startswith('aq_') and col != 'aq_total']

# Other features (demographics, education, occupation, etc.)
other_features = [col for col in x.columns if col not in demographic_features + total_scores + engineered_features + spq_items + eq_items + sqr_items + aq_items]

print("1. DEMOGRAPHIC FEATURES:")
for feat in demographic_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("2. TOTAL SCORES:")
for feat in total_scores:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("3. INDIVIDUAL QUESTIONNAIRE ITEMS:")
print("   SPQ items (10):", ", ".join(spq_items))
print("   EQ items (10):", ", ".join(eq_items))
print("   SQR items (10):", ", ".join(sqr_items))
print("   AQ items (10):", ", ".join(aq_items))
print()

print("4. ENGINEERED FEATURES:")
for feat in engineered_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("5. OTHER FEATURES:")
for feat in other_features:
    print(f"   - {feat}")
print()

print("="*60)
print("FEATURE SUMMARY")
print("="*60)
print(f"Demographic features: {len([f for f in demographic_features if f in x.columns])}")
print(f"Total scores: {len([f for f in total_scores if f in x.columns])}")
print(f"Individual SPQ items: {len(spq_items)}")
print(f"Individual EQ items: {len(eq_items)}")
print(f"Individual SQR items: {len(sqr_items)}")
print(f"Individual AQ items: {len(aq_items)}")
print(f"Engineered features: {len([f for f in engineered_features if f in x.columns])}")
print(f"Other features: {len(other_features)}")
print(f"TOTAL: {len(x.columns)} features")

# Check for problematic features
print("\n" + "="*60)
print("PROBLEMATIC FEATURES CHECK")
print("="*60)
problematic_features = ['userid', 'repeat', 'handedness', 'country_region']
for feat in problematic_features:
    if feat in x.columns:
        print(f"{feat} - SHOULD BE REMOVED")
    else:
        print(f"{feat} - NOT PRESENT")

# Check sex features specifically
print("\n" + "="*60)
print("SEX FEATURES CHECK")
print("="*60)
sex_features = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']
for feat in sex_features:
    if feat in x.columns:
        print(f"{feat} - PRESENT")
    else:
        print(f"{feat} - MISSING")

# re running baseline models on cleaned and correct features dataset 

In [ ]:
# BASELINE MODELS WITH CLEANED PROPERLY BALANCED DATASET
print("="*60)
print("BASELINE MODELS WITH CLEANED PROPERLY BALANCED DATASET")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load the cleaned properly balanced engineered dataset
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
print(f"Loaded cleaned properly balanced engineered dataset: {df.shape}")

# Verify we have the correct features
print("\nVerifying feature set...")
expected_features = [
    # Demographics (6)
    'age', 'sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0', 'is_stem_occupation',
    # Total scores (5)
    'spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score',
    # Individual SPQ items (10)
    'spq_1', 'spq_2', 'spq_3', 'spq_4', 'spq_5', 'spq_6', 'spq_7', 'spq_8', 'spq_9', 'spq_10',
    # Individual EQ items (10)
    'eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5', 'eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10',
    # Individual SQR items (10)
    'sqr_1', 'sqr_2', 'sqr_3', 'sqr_4', 'sqr_5', 'sqr_6', 'sqr_7', 'sqr_8', 'sqr_9', 'sqr_10',
    # Individual AQ items (10)
    'aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10',
    # Engineered features (13)
    'log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 'age_x_eq', 
    'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq', 'age_group_19-30', 
    'age_group_31-45', 'age_group_46-60', 'age_group_61+'
]

actual_features = [col for col in df.columns if col != 'autism_target']
missing_features = [feat for feat in expected_features if feat not in actual_features]
extra_features = [feat for feat in actual_features if feat not in expected_features]

print(f"Expected features: {len(expected_features)}")
print(f"Actual features: {len(actual_features)}")
print(f"Missing features: {len(missing_features)}")
print(f"Extra features: {len(extra_features)}")

if missing_features:
    print(f"Missing: {missing_features}")
if extra_features:
    print(f"Extra: {extra_features}")

if len(missing_features) == 0 and len(extra_features) == 0:
    print("Feature set matches exactly!")
else:
    print("Feature set does not match - check the dataset")

# Prepare features and target
x = df.drop(columns=['autism_target'])
y = df['autism_target']

print(f"\nFeatures shape: {x.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle missing values
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Scale features for Logistic Regression only
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x_imputed), columns=x_imputed.columns)

# Split data
x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)
x_train_scaled, x_val_scaled, y_train_scaled, y_val_scaled = train_test_split(x_scaled, y, stratify=y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")

# Store all models and their results
models = {}
results = {}

# 1. Logistic Regression
print("\n1. Training Logistic Regression...")
logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logreg.fit(x_train_scaled, y_train_scaled)
logreg_pred = logreg.predict(x_val_scaled)
logreg_proba = logreg.predict_proba(x_val_scaled)[:, 1]
logreg_auc = roc_auc_score(y_val_scaled, logreg_proba)
logreg_f1 = f1_score(y_val_scaled, logreg_pred)

models['logreg'] = logreg
results['logreg'] = {'auc': logreg_auc, 'f1': logreg_f1, 'model': logreg, 'data': 'scaled'}

print(f"Logistic Regression - ROC-AUC: {logreg_auc:.4f}, F1: {logreg_f1:.4f}")

# 2. Random Forest
print("\n2. Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_val)
rf_proba = rf.predict_proba(x_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_proba)
rf_f1 = f1_score(y_val, rf_pred)

models['rf'] = rf
results['rf'] = {'auc': rf_auc, 'f1': rf_f1, 'model': rf, 'data': 'unscaled'}

print(f"Random Forest - ROC-AUC: {rf_auc:.4f}, F1: {rf_f1:.4f}")

# 3. XGBoost
print("\n3. Training XGBoost...")
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_val)
xgb_proba = xgb.predict_proba(x_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_proba)
xgb_f1 = f1_score(y_val, xgb_pred)

models['xgb'] = xgb
results['xgb'] = {'auc': xgb_auc, 'f1': xgb_f1, 'model': xgb, 'data': 'unscaled'}

print(f"XGBoost - ROC-AUC: {xgb_auc:.4f}, F1: {xgb_f1:.4f}")

# 4. LightGBM
print("\n4. Training LightGBM...")
lgb = LGBMClassifier(random_state=42, verbose=-1)
lgb.fit(x_train, y_train)
lgb_pred = lgb.predict(x_val)
lgb_proba = lgb.predict_proba(x_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_proba)
lgb_f1 = f1_score(y_val, lgb_pred)

models['lgb'] = lgb
results['lgb'] = {'auc': lgb_auc, 'f1': lgb_f1, 'model': lgb, 'data': 'unscaled'}

print(f"LightGBM - ROC-AUC: {lgb_auc:.4f}, F1: {lgb_f1:.4f}")

# 5. Gradient Boosting
print("\n5. Training Gradient Boosting...")
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
gb_pred = gb.predict(x_val)
gb_proba = gb.predict_proba(x_val)[:, 1]
gb_auc = roc_auc_score(y_val, gb_proba)
gb_f1 = f1_score(y_val, gb_pred)

models['gb'] = gb
results['gb'] = {'auc': gb_auc, 'f1': gb_f1, 'model': gb, 'data': 'unscaled'}

print(f"Gradient Boosting - ROC-AUC: {gb_auc:.4f}, F1: {gb_f1:.4f}")

# 6. Extra Trees
print("\n6. Training Extra Trees...")
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(x_train, y_train)
et_pred = et.predict(x_val)
et_proba = et.predict_proba(x_val)[:, 1]
et_auc = roc_auc_score(y_val, et_proba)
et_f1 = f1_score(y_val, et_pred)

models['et'] = et
results['et'] = {'auc': et_auc, 'f1': et_f1, 'model': et, 'data': 'unscaled'}

print(f"Extra Trees - ROC-AUC: {et_auc:.4f}, F1: {et_f1:.4f}")

# 7. AdaBoost
print("\n7. Training AdaBoost...")
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(x_train, y_train)
ada_pred = ada.predict(x_val)
ada_proba = ada.predict_proba(x_val)[:, 1]
ada_auc = roc_auc_score(y_val, ada_proba)
ada_f1 = f1_score(y_val, ada_pred)

models['ada'] = ada
results['ada'] = {'auc': ada_auc, 'f1': ada_f1, 'model': ada, 'data': 'unscaled'}

print(f"AdaBoost - ROC-AUC: {ada_auc:.4f}, F1: {ada_f1:.4f}")

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k]['auc'])
best_model = results[best_model_name]['model']
best_auc = results[best_model_name]['auc']
best_f1 = results[best_model_name]['f1']

print(f"\n{'='*50}")
print(f"BEST MODEL: {best_model_name.upper()}")
print(f"ROC-AUC: {best_auc:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"{'='*50}")

# Store best model for later use
best_model_name_final = best_model_name

# Show all results in a summary table
print(f"\n{'='*60}")
print("SUMMARY OF ALL MODEL PERFORMANCES")
print("="*60)
print(f"{'Model':<15} {'ROC-AUC':<10} {'F1-Score':<10}")
print("-" * 35)
for model_name, result in results.items():
    print(f"{model_name.upper():<15} {result['auc']:<10.4f} {result['f1']:<10.4f}")
print("="*60)

# quick print/check features
- all good

In [ ]:
# Print features being used in the new model and dataset
print("="*60)
print("ALL FEATURES USED BY THE MODEL (NEW DATASET)")
print("="*60)

# Load the dataset and prepare features
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
x = df.drop(columns=['autism_target'])

print(f"Total number of features: {len(x.columns)}")
print()

# Group features by category
demographic_features = ['age', 'sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0', 'is_stem_occupation']
total_scores = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                      'age_x_eq', 'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq',
                      'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+']

# Individual questionnaire items
spq_items = [col for col in x.columns if col.startswith('spq_') and col != 'spq_total']
eq_items = [col for col in x.columns if col.startswith('eq_') and col != 'eq_total']
sqr_items = [col for col in x.columns if col.startswith('sqr_') and col != 'sqr_total']
aq_items = [col for col in x.columns if col.startswith('aq_') and col != 'aq_total']

print("1. DEMOGRAPHIC FEATURES:")
for feat in demographic_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("2. TOTAL SCORES:")
for feat in total_scores:
    if feat in x.columns:
        print(f"   - {feat}")
print()

print("3. INDIVIDUAL QUESTIONNAIRE ITEMS:")
print("   SPQ items (10):", ", ".join(spq_items))
print("   EQ items (10):", ", ".join(eq_items))
print("   SQR items (10):", ", ".join(sqr_items))
print("   AQ items (10):", ", ".join(aq_items))
print()

print("4. ENGINEERED FEATURES:")
for feat in engineered_features:
    if feat in x.columns:
        print(f"   - {feat}")
print()

# Check for any other features not categorized
other_features = [col for col in x.columns if col not in demographic_features + total_scores + engineered_features + spq_items + eq_items + sqr_items + aq_items]
if other_features:
    print("5. OTHER FEATURES:")
    for feat in other_features:
        print(f"   - {feat}")
    print()

print("="*60)
print("FEATURE SUMMARY")
print("="*60)
print(f"Demographic features: {len([f for f in demographic_features if f in x.columns])}")
print(f"Total scores: {len([f for f in total_scores if f in x.columns])}")
print(f"Individual SPQ items: {len(spq_items)}")
print(f"Individual EQ items: {len(eq_items)}")
print(f"Individual SQR items: {len(sqr_items)}")
print(f"Individual AQ items: {len(aq_items)}")
print(f"Engineered features: {len([f for f in engineered_features if f in x.columns])}")
if other_features:
    print(f"Other features: {len(other_features)}")
print(f"TOTAL: {len(x.columns)} features")

# Compare with original feature set
print("\n" + "="*60)
print("COMPARISON WITH ORIGINAL FEATURE SET")
print("="*60)

# Check for differences in sex features
original_sex = ['sex_2.0', 'sex_3.0', 'sex_4.0', 'sex_unknown']
new_sex = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']

print("Sex features comparison:")
print(f"Original: {original_sex}")
print(f"New: {new_sex}")

# Check for missing features
missing_from_new = [feat for feat in original_sex if feat not in new_sex]
extra_in_new = [feat for feat in new_sex if feat not in original_sex]

if missing_from_new:
    print(f"Missing from new: {missing_from_new}")
if extra_in_new:
    print(f"Extra in new: {extra_in_new}")

if not missing_from_new and not extra_in_new:
    print("Sex features match exactly")
else:
    print("Sex features differ - this is expected due to proper one-hot encoding")

# threshold tuning

In [ ]:
# THRESHOLD OPTIMIZATION FOR CLEANED PROPERLY BALANCED DATASET
print("="*60)
print("THRESHOLD OPTIMIZATION FOR CLEANED PROPERLY BALANCED DATASET")
print("="*60)

from sklearn.metrics import precision_recall_curve, f1_score, classification_report, roc_auc_score

# Dictionary to store optimized thresholds and scores
optimized_results = {}

# Iterate through all models
for model_name, model_info in results.items():
    model = model_info['model']
    
    # Get predictions based on whether the model used scaled data or not
    if model_info['data'] == 'scaled':
        probs = model.predict_proba(x_val_scaled)[:, 1]
        x_val_current = x_val_scaled
        y_val_current = y_val
    else:
        probs = model.predict_proba(x_val)[:, 1]
        x_val_current = x_val
        y_val_current = y_val
    
    # Find best threshold for F1 score
    prec, rec, thresholds = precision_recall_curve(y_val_current, probs)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    
    # Handle case where thresholds might be empty
    if len(thresholds) > 0:
        best_thresh = thresholds[np.argmax(f1_scores[:-1])]
    else:
        best_thresh = 0.5
    
    # Evaluate at best threshold
    pred_thresh = (probs >= best_thresh).astype(int)
    optimized_f1 = f1_score(y_val_current, pred_thresh)
    optimized_auc = roc_auc_score(y_val_current, probs)
    
    # Store results
    optimized_results[model_name] = {
        'threshold': best_thresh,
        'f1': optimized_f1,
        'auc': optimized_auc,
        'probs': probs,
        'x_val': x_val_current,
        'y_val': y_val_current
    }
    
    # Print results
    print(f"\n{model_name.upper()} - Best threshold for F1: {best_thresh:.3f}")
    print(f"F1 at best threshold: {optimized_f1:.4f}")
    print(f"ROC-AUC: {optimized_auc:.4f}")
    print(classification_report(y_val_current, pred_thresh))

# Find best model after threshold optimization
best_model_name_optimized = max(optimized_results.keys(), key=lambda k: optimized_results[k]['f1'])
best_thresh_optimized = optimized_results[best_model_name_optimized]['threshold']
best_f1_optimized = optimized_results[best_model_name_optimized]['f1']
best_auc_optimized = optimized_results[best_model_name_optimized]['auc']

print(f"\n{'='*50}")
print(f"BEST MODEL AFTER THRESHOLD OPTIMIZATION: {best_model_name_optimized.upper()}")
print(f"Best threshold: {best_thresh_optimized:.3f}")
print(f"F1-Score: {best_f1_optimized:.4f}")
print(f"ROC-AUC: {best_auc_optimized:.4f}")
print(f"{'='*50}")

# Store best model and data for feature importance
best_model_name_final = best_model_name_optimized
best_model_final = models[best_model_name_final]
best_x_val_final = optimized_results[best_model_name_final]['x_val']
best_y_val_final = optimized_results[best_model_name_final]['y_val']

# Show all results in a summary table
print(f"\n{'='*60}")
print("SUMMARY OF ALL MODEL PERFORMANCES AFTER THRESHOLD OPTIMIZATION")
print("="*60)
print(f"{'Model':<15} {'Threshold':<12} {'F1-Score':<10} {'ROC-AUC':<10}")
print("-" * 47)
for model_name, result in optimized_results.items():
    print(f"{model_name.upper():<15} {result['threshold']:<12.3f} {result['f1']:<10.4f} {result['auc']:<10.4f}")
print("="*60)

# Compare with baseline performance
print(f"\n{'='*60}")
print("PERFORMANCE COMPARISON: BASELINE vs OPTIMIZED")
print("="*60)
print(f"{'Model':<15} {'Baseline F1':<12} {'Optimized F1':<12} {'Improvement':<12}")
print("-" * 51)
for model_name in results.keys():
    baseline_f1 = results[model_name]['f1']
    optimized_f1 = optimized_results[model_name]['f1']
    improvement = optimized_f1 - baseline_f1
    print(f"{model_name.upper():<15} {baseline_f1:<12.4f} {optimized_f1:<12.4f} {improvement:<12.4f}")
print("="*60)

# Show best model details
print(f"\n{'='*60}")
print(f"DETAILED RESULTS FOR BEST MODEL: {best_model_name_optimized.upper()}")
print("="*60)
print(f"Best threshold: {best_thresh_optimized:.3f}")
print(f"F1-Score: {best_f1_optimized:.4f}")
print(f"ROC-AUC: {best_auc_optimized:.4f}")
print(f"Model type: {type(best_model_final).__name__}")
print(f"Data scaling: {results[best_model_name_optimized]['data']}")
print("="*60)

# feature importance 

In [ ]:
# FEATURE IMPORTANCE ANALYSIS FOR CLEANED PROPERLY BALANCED DATASET
print("="*60)
print("FEATURE IMPORTANCE ANALYSIS FOR CLEANED PROPERLY BALANCED DATASET")
print("="*60)

import pandas as pd
from sklearn.inspection import permutation_importance

print(f"Feature Importance Analysis for {best_model_name_final.upper()} (Best Model)")
print("="*60)

# Get feature importance based on model type
if hasattr(best_model_final, 'feature_importances_'):
    # Tree-based models (RF, XGB, LGB, GB)
    importances = pd.Series(best_model_final.feature_importances_, index=x_train.columns)
    print(f"\nTop 20 features by {best_model_name_final.upper()} importance:")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
else:
    # Linear models (LogReg, SVM)
    if hasattr(best_model_final, 'coef_'):
        # Logistic Regression
        importances = pd.Series(np.abs(best_model_final.coef_[0]), index=x_train.columns)
    else:
        # SVM or other models
        importances = pd.Series(np.zeros(len(x_train.columns)), index=x_train.columns)
    
    print(f"\nTop 20 features by {best_model_name_final.upper()} coefficients (absolute values):")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Coefficient': importances
    }).sort_values('Coefficient', ascending=False)

# Permutation importance (more robust, works for all models)
print(f"\nComputing permutation importance for {best_model_name_final.upper()}...")
perm_importance = permutation_importance(
    best_model_final, 
    best_x_val_final, 
    best_y_val_final, 
    n_repeats=5, 
    random_state=42
)

perm_importance_df = pd.DataFrame({
    'Feature': x_train.columns,
    'Permutation_importance': perm_importance.importances_mean
}).sort_values('Permutation_importance', ascending=False)

print(f"\nTop 20 features by Permutation importance ({best_model_name_final.upper()}):")
print(perm_importance_df.head(20))

# Save both importance measures
print(f"\nSaving feature importance results for {best_model_name_final.upper()}...")
importance_df.to_csv(f'feature_importance_{best_model_name_final}_cleaned.csv', index=False)
perm_importance_df.to_csv(f'permutation_importance_{best_model_name_final}_cleaned.csv', index=False)

print(f"\nFeature importance analysis complete for {best_model_name_final.upper()}")
print(f"Results saved to: feature_importance_{best_model_name_final}_cleaned.csv")
print(f"Permutation importance saved to: permutation_importance_{best_model_name_final}_cleaned.csv")

# Check sex feature importance specifically
print(f"\n{'='*50}")
print("SEX FEATURE IMPORTANCE ANALYSIS")
print("="*50)

sex_features = ['sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0']
for feat in sex_features:
    if feat in x_train.columns:
        if hasattr(best_model_final, 'feature_importances_'):
            importance = importances[feat]
            print(f"{feat}: {importance:.6f}")
        else:
            coef = importances[feat]
            print(f"{feat}: {coef:.6f}")
        
        # Check permutation importance
        perm_imp = perm_importance_df[perm_importance_df['Feature'] == feat]['Permutation_importance'].values
        if len(perm_imp) > 0:
            print(f"  Permutation importance: {perm_imp[0]:.6f}")

# Check feature importance by category
print(f"\n{'='*50}")
print("FEATURE IMPORTANCE BY CATEGORY")
print("="*50)

# Define feature categories
demographic_features = ['age', 'sex_1.0', 'sex_2.0', 'sex_3.0', 'sex_4.0', 'is_stem_occupation']
total_scores = ['spq_total', 'eq_total', 'sqr_total', 'aq_total', 'd_score']
aq_items = [col for col in x_train.columns if col.startswith('aq_') and col != 'aq_total']
spq_items = [col for col in x_train.columns if col.startswith('spq_') and col != 'spq_total']
eq_items = [col for col in x_train.columns if col.startswith('eq_') and col != 'eq_total']
sqr_items = [col for col in x_train.columns if col.startswith('sqr_') and col != 'sqr_total']
engineered_features = ['log_aq_total', 'sqrt_age', 'aq_eq_interaction', 'sqp_aq_interaction', 
                      'age_x_eq', 'age_x_aq', 'aq_spq_ratio', 'eq_sqr_ratio', 'high_aq',
                      'age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+']

# Calculate total importance by category
categories = {
    'Demographics': demographic_features,
    'Total Scores': total_scores,
    'AQ Items': aq_items,
    'SPQ Items': spq_items,
    'EQ Items': eq_items,
    'SQR Items': sqr_items,
    'Engineered Features': engineered_features
}

print("Total importance by category:")
for category, features in categories.items():
    category_features = [f for f in features if f in x_train.columns]
    if category_features:
        if hasattr(best_model_final, 'feature_importances_'):
            total_importance = importances[category_features].sum()
        else:
            total_importance = importances[category_features].sum()
        print(f"{category}: {total_importance:.4f}")

# Show top features by category
print(f"\n{'='*50}")
print("TOP FEATURES BY CATEGORY")
print("="*50)

for category, features in categories.items():
    category_features = [f for f in features if f in x_train.columns]
    if category_features:
        category_importances = importances[category_features].sort_values(ascending=False)
        print(f"\n{category} (Top 5):")
        for feat, imp in category_importances.head(5).items():
            print(f"  {feat}: {imp:.6f}")

# Summary statistics
print(f"\n{'='*50}")
print("FEATURE IMPORTANCE SUMMARY")
print("="*50)
print(f"Total features analyzed: {len(x_train.columns)}")
print(f"Highest importance: {importances.max():.6f}")
print(f"Lowest importance: {importances.min():.6f}")
print(f"Mean importance: {importances.mean():.6f}")
print(f"Sex features combined importance: {importances[sex_features].sum():.6f}")
print(f"Sex features rank: {importances[sex_features].sum() / importances.sum() * 100:.1f}% of total importance")

# some quick experiments to try and improve performance 
- advanced feature engineering (optimised) - not really more needed
- data aug - pointless as dataset already balanced 
- emsemble 
- advanced models (cat boost)
- threshold optimisation 


In [ ]:
# ADVANCED MODELS - OPTIMIZED FOR LAPTOP (SKIP SVM)
print("="*60)
print("ADVANCED MODELS - OPTIMIZED FOR LAPTOP (SKIP SVM)")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the advanced feature engineered dataset
df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
print(f"Dataset shape: {df.shape}")

# Prepare features and target
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
print("\nHandling missing values...")
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Scale features
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x_imputed), columns=x_imputed.columns)

# Split data
x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)
x_train_scaled, x_val_scaled, y_train_scaled, y_val_scaled = train_test_split(x_scaled, y, stratify=y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")

# Store results
advanced_results = {}

# 1. CATBOOST (often outperforms LightGBM/XGBoost)
print("\n1. Training CatBoost...")
try:
    from catboost import CatBoostClassifier
    
    catboost_model = CatBoostClassifier(
        iterations=200,  # Reduced for speed
        learning_rate=0.1,
        depth=6,
        l2_leaf_reg=3,
        random_state=42,
        verbose=False,
        task_type='CPU'
    )
    
    catboost_model.fit(x_train, y_train)
    catboost_pred = catboost_model.predict(x_val)
    catboost_proba = catboost_model.predict_proba(x_val)[:, 1]
    catboost_auc = roc_auc_score(y_val, catboost_proba)
    catboost_f1 = f1_score(y_val, catboost_pred)
    
    advanced_results['catboost'] = {
        'auc': catboost_auc,
        'f1': catboost_f1,
        'model': catboost_model,
        'data': 'unscaled'
    }
    
    print(f"CatBoost - ROC-AUC: {catboost_auc:.4f}, F1: {catboost_f1:.4f}")
    
except ImportError:
    print("CatBoost not installed. Install with: pip install catboost")

# 2. NEURAL NETWORK (MLP) - Fast
print("\n2. Training Neural Network...")
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(50, 25),  # Smaller for speed
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate='adaptive',
    max_iter=200,  # Reduced for speed
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp.fit(x_train_scaled, y_train_scaled)
mlp_pred = mlp.predict(x_val_scaled)
mlp_proba = mlp.predict_proba(x_val_scaled)[:, 1]
mlp_auc = roc_auc_score(y_val_scaled, mlp_proba)
mlp_f1 = f1_score(y_val_scaled, mlp_pred)

advanced_results['mlp'] = {
    'auc': mlp_auc,
    'f1': mlp_f1,
    'model': mlp,
    'data': 'scaled'
}

print(f"Neural Network - ROC-AUC: {mlp_auc:.4f}, F1: {mlp_f1:.4f}")

# 3. NAIVE BAYES - Very Fast
print("\n3. Training Naive Bayes...")
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(x_train, y_train)
nb_pred = nb.predict(x_val)
nb_proba = nb.predict_proba(x_val)[:, 1]
nb_auc = roc_auc_score(y_val, nb_proba)
nb_f1 = f1_score(y_val, nb_pred)

advanced_results['naive_bayes'] = {
    'auc': nb_auc,
    'f1': nb_f1,
    'model': nb,
    'data': 'unscaled'
}

print(f"Naive Bayes - ROC-AUC: {nb_auc:.4f}, F1: {nb_f1:.4f}")

# 4. K-NEAREST NEIGHBORS - Fast
print("\n4. Training KNN...")
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, weights='uniform')
knn.fit(x_train_scaled, y_train_scaled)
knn_pred = knn.predict(x_val_scaled)
knn_proba = knn.predict_proba(x_val_scaled)[:, 1]
knn_auc = roc_auc_score(y_val_scaled, knn_proba)
knn_f1 = f1_score(y_val_scaled, knn_pred)

advanced_results['knn'] = {
    'auc': knn_auc,
    'f1': knn_f1,
    'model': knn,
    'data': 'scaled'
}

print(f"KNN - ROC-AUC: {knn_auc:.4f}, F1: {knn_f1:.4f}")

# 5. DECISION TREE - Fast
print("\n5. Training Decision Tree...")
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, max_depth=10)
dt.fit(x_train, y_train)
dt_pred = dt.predict(x_val)
dt_proba = dt.predict_proba(x_val)[:, 1]
dt_auc = roc_auc_score(y_val, dt_proba)
dt_f1 = f1_score(y_val, dt_pred)

advanced_results['decision_tree'] = {
    'auc': dt_auc,
    'f1': dt_f1,
    'model': dt,
    'data': 'unscaled'
}

print(f"Decision Tree - ROC-AUC: {dt_auc:.4f}, F1: {dt_f1:.4f}")

# Find best advanced model
if advanced_results:
    best_advanced_model_name = max(advanced_results.keys(), key=lambda k: advanced_results[k]['auc'])
    best_advanced_model = advanced_results[best_advanced_model_name]['model']
    best_advanced_auc = advanced_results[best_advanced_model_name]['auc']
    best_advanced_f1 = advanced_results[best_advanced_model_name]['f1']

    print(f"\n{'='*50}")
    print(f"BEST ADVANCED MODEL: {best_advanced_model_name.upper()}")
    print(f"ROC-AUC: {best_advanced_auc:.4f}")
    print(f"F1-Score: {best_advanced_f1:.4f}")
    print(f"{'='*50}")

    # Show all results
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL ADVANCED MODEL PERFORMANCES")
    print("="*60)
    print(f"{'Model':<15} {'ROC-AUC':<10} {'F1-Score':<10}")
    print("-" * 35)
    for model_name, result in advanced_results.items():
        print(f"{model_name.upper():<15} {result['auc']:<10.4f} {result['f1']:<10.4f}")
    print("="*60)

print("\n" + "="*60)
print("ADVANCED MODELS COMPLETE")
print("="*60)

In [ ]:
# ENSEMBLE METHODS - OPTIMIZED FOR LAPTOP (FIXED)
print("="*60)
print("ENSEMBLE METHODS - OPTIMIZED FOR LAPTOP (FIXED)")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

# Load the advanced feature engineered dataset
try:
    df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
    print("Loaded advanced feature engineered dataset")
except:
    df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
    print("Loaded original dataset (advanced features not found)")

print(f"Dataset shape: {df.shape}")

# Prepare features and target
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
print("\nHandling missing values...")
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Split data
x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")

# Store results
ensemble_results = {}

# Define base estimators (optimized for speed)
estimators = [
    ('lgb', LGBMClassifier(random_state=42, verbose=-1, n_estimators=100)),
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=100)),
    ('rf', RandomForestClassifier(random_state=42, n_estimators=100)),
    ('gb', GradientBoostingClassifier(random_state=42, n_estimators=100))
]

# 1. VOTING CLASSIFIER (HARD VOTING)
print("\n1. Training Voting Classifier (Hard)...")
voting_hard = VotingClassifier(
    estimators=estimators,
    voting='hard'
)

voting_hard.fit(x_train, y_train)
voting_hard_pred = voting_hard.predict(x_val)
voting_hard_auc = roc_auc_score(y_val, voting_hard_pred)  # Note: no probabilities for hard voting
voting_hard_f1 = f1_score(y_val, voting_hard_pred)

ensemble_results['voting_hard'] = {
    'auc': voting_hard_auc,
    'f1': voting_hard_f1,
    'model': voting_hard,
    'data': 'unscaled'
}

print(f"Voting (Hard) - ROC-AUC: {voting_hard_auc:.4f}, F1: {voting_hard_f1:.4f}")

# 2. VOTING CLASSIFIER (SOFT VOTING)
print("\n2. Training Voting Classifier (Soft)...")
voting_soft = VotingClassifier(
    estimators=estimators,
    voting='soft'
)

voting_soft.fit(x_train, y_train)
voting_soft_pred = voting_soft.predict(x_val)
voting_soft_proba = voting_soft.predict_proba(x_val)[:, 1]
voting_soft_auc = roc_auc_score(y_val, voting_soft_proba)
voting_soft_f1 = f1_score(y_val, voting_soft_pred)

ensemble_results['voting_soft'] = {
    'auc': voting_soft_auc,
    'f1': voting_soft_f1,
    'model': voting_soft,
    'data': 'unscaled'
}

print(f"Voting (Soft) - ROC-AUC: {voting_soft_auc:.4f}, F1: {voting_soft_f1:.4f}")

# 3. STACKING CLASSIFIER
print("\n3. Training Stacking Classifier...")
stacking_classifier = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=3,  # Reduced for speed
    stack_method='predict_proba'
)

stacking_classifier.fit(x_train, y_train)
stacking_pred = stacking_classifier.predict(x_val)
stacking_proba = stacking_classifier.predict_proba(x_val)[:, 1]
stacking_auc = roc_auc_score(y_val, stacking_proba)
stacking_f1 = f1_score(y_val, stacking_pred)

ensemble_results['stacking'] = {
    'auc': stacking_auc,
    'f1': stacking_f1,
    'model': stacking_classifier,
    'data': 'unscaled'
}

print(f"Stacking - ROC-AUC: {stacking_auc:.4f}, F1: {stacking_f1:.4f}")

# 4. WEIGHTED ENSEMBLE (Manual)
print("\n4. Training Weighted Ensemble...")
# Train individual models
models = {}
for name, (est_name, estimator) in enumerate(estimators):
    estimator.fit(x_train, y_train)
    models[est_name] = estimator

# Get predictions from all models
predictions = {}
probabilities = {}
for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(x_val)[:, 1]
        pred = (proba > 0.5).astype(int)
    else:
        pred = model.predict(x_val)
        proba = pred.astype(float)
    
    predictions[name] = pred
    probabilities[name] = proba

# Weighted average (equal weights for now)
weights = {'lgb': 0.3, 'xgb': 0.3, 'rf': 0.2, 'gb': 0.2}
weighted_proba = np.zeros(len(y_val))

for name, weight in weights.items():
    weighted_proba += probabilities[name] * weight

weighted_pred = (weighted_proba > 0.5).astype(int)
weighted_auc = roc_auc_score(y_val, weighted_proba)
weighted_f1 = f1_score(y_val, weighted_pred)

ensemble_results['weighted'] = {
    'auc': weighted_auc,
    'f1': weighted_f1,
    'model': models,
    'data': 'unscaled',
    'weights': weights
}

print(f"Weighted Ensemble - ROC-AUC: {weighted_auc:.4f}, F1: {weighted_f1:.4f}")

# Find best ensemble model
best_ensemble_model_name = max(ensemble_results.keys(), key=lambda k: ensemble_results[k]['auc'])
best_ensemble_model = ensemble_results[best_ensemble_model_name]['model']
best_ensemble_auc = ensemble_results[best_ensemble_model_name]['auc']
best_ensemble_f1 = ensemble_results[best_ensemble_model_name]['f1']

print(f"\n{'='*50}")
print(f"BEST ENSEMBLE MODEL: {best_ensemble_model_name.upper()}")
print(f"ROC-AUC: {best_ensemble_auc:.4f}")
print(f"F1-Score: {best_ensemble_f1:.4f}")
print(f"{'='*50}")

# Show all results
print(f"\n{'='*60}")
print("SUMMARY OF ALL ENSEMBLE MODEL PERFORMANCES")
print("="*60)
print(f"{'Model':<15} {'ROC-AUC':<10} {'F1-Score':<10}")
print("-" * 35)
for model_name, result in ensemble_results.items():
    print(f"{model_name.upper():<15} {result['auc']:<10.4f} {result['f1']:<10.4f}")
print("="*60)

print("\n" + "="*60)
print("ENSEMBLE METHODS COMPLETE")
print("="*60)

In [ ]:
# THRESHOLD OPTIMIZATION STRATEGY - OPTIMIZED FOR LAPTOP (FIXED)
print("="*60)
print("THRESHOLD OPTIMIZATION STRATEGY - OPTIMIZED FOR LAPTOP (FIXED)")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Load the best model from previous cells (or train a new one)
try:
    df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
    print("Loaded advanced feature engineered dataset")
except:
    df = pd.read_csv('data/processed/data_c4_properly_balanced_fe.csv')
    print("Loaded original dataset")

print(f"Dataset shape: {df.shape}")

# Prepare features and target
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
print("\nHandling missing values...")
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Split data
x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Validation set: {x_val.shape}")

# Train a model for threshold optimization
print("\nTraining model for threshold optimization...")
model = LGBMClassifier(random_state=42, verbose=-1, n_estimators=100)
model.fit(x_train, y_train)

# Get predictions
y_proba = model.predict_proba(x_val)[:, 1]

# Define optimization function
def optimize_threshold(y_true, y_proba, metric='f1'):
    """
    Optimize threshold for different metrics
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    scores = []
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        if metric == 'f1':
            score = f1_score(y_true, y_pred)
        elif metric == 'precision':
            score = precision_score(y_true, y_pred)
        elif metric == 'recall':
            score = recall_score(y_true, y_pred)
        elif metric == 'balanced_accuracy':
            score = (precision_score(y_true, y_pred) + recall_score(y_true, y_pred)) / 2
        scores.append(score)
    
    best_thresh = thresholds[np.argmax(scores)]
    return best_thresh, max(scores)

# 1. OPTIMIZE FOR DIFFERENT METRICS
print("\n1. Optimizing thresholds for different metrics...")
metrics = ['f1', 'precision', 'recall', 'balanced_accuracy']
threshold_results = {}

for metric in metrics:
    best_thresh, best_score = optimize_threshold(y_val, y_proba, metric)
    threshold_results[metric] = {
        'threshold': best_thresh,
        'score': best_score
    }
    print(f"{metric.upper()}: Best threshold = {best_thresh:.3f}, Score = {best_score:.4f}")

# 2. CUSTOM METRICS
print("\n2. Optimizing for custom metrics...")

# F1 with different beta values
def f1_beta_score(y_true, y_pred, beta=2):
    """F1 score with custom beta (beta > 1 gives more weight to recall)"""
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    if precision + recall == 0:
        return 0
    return (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall)

# Optimize for F1 with beta=2 (more weight to recall)
thresholds = np.arange(0.1, 0.9, 0.01)
f1_beta_scores = []

for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    score = f1_beta_score(y_val, y_pred, beta=2)
    f1_beta_scores.append(score)

best_thresh_f1_beta = thresholds[np.argmax(f1_beta_scores)]
best_score_f1_beta = max(f1_beta_scores)

threshold_results['f1_beta_2'] = {
    'threshold': best_thresh_f1_beta,
    'score': best_score_f1_beta
}

print(f"F1-Beta(2): Best threshold = {best_thresh_f1_beta:.3f}, Score = {best_score_f1_beta:.4f}")

# 3. BUSINESS METRICS
print("\n3. Optimizing for business metrics...")

# Cost-based optimization (assuming false positive costs more than false negative)
def business_score(y_true, y_pred, fp_cost=2, fn_cost=1):
    """Business score considering different costs for FP and FN"""
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    total_cost = fp * fp_cost + fn * fn_cost
    return -total_cost  # Negative because we want to minimize cost

thresholds = np.arange(0.1, 0.9, 0.01)
business_scores = []

for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    score = business_score(y_val, y_pred, fp_cost=2, fn_cost=1)
    business_scores.append(score)

best_thresh_business = thresholds[np.argmax(business_scores)]
best_score_business = max(business_scores)

threshold_results['business'] = {
    'threshold': best_thresh_business,
    'score': best_score_business
}

print(f"Business: Best threshold = {best_thresh_business:.3f}, Score = {best_score_business:.4f}")

# 4. COMPARISON OF ALL THRESHOLDS
print("\n4. Comparing all threshold optimizations...")
print(f"{'Metric':<15} {'Threshold':<12} {'Score':<10} {'Precision':<10} {'Recall':<10}")
print("-" * 65)

for metric, result in threshold_results.items():
    thresh = result['threshold']
    y_pred = (y_proba >= thresh).astype(int)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    score = result['score']
    
    print(f"{metric.upper():<15} {thresh:<12.3f} {score:<10.4f} {precision:<10.4f} {recall:<10.4f}")

# 5. THRESHOLD ANALYSIS
print("\n5. Threshold analysis summary...")
print("="*60)

# Find the most balanced threshold (closest precision and recall)
thresholds = np.arange(0.1, 0.9, 0.01)
balanced_scores = []

for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    balance = 1 - abs(precision - recall)  # Higher is more balanced
    balanced_scores.append(balance)

best_thresh_balanced = thresholds[np.argmax(balanced_scores)]
best_balance = max(balanced_scores)

print(f"Most balanced threshold: {best_thresh_balanced:.3f} (balance score: {best_balance:.4f})")

# Show ROC-AUC for reference
roc_auc = roc_auc_score(y_val, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

print("\n" + "="*60)
print("THRESHOLD OPTIMIZATION COMPLETE")
print("="*60)
print("Recommended thresholds:")
print(f"  - For maximum F1: {threshold_results['f1']['threshold']:.3f}")
print(f"  - For maximum precision: {threshold_results['precision']['threshold']:.3f}")
print(f"  - For maximum recall: {threshold_results['recall']['threshold']:.3f}")
print(f"  - For balanced performance: {best_thresh_balanced:.3f}")
print(f"  - For business optimization: {threshold_results['business']['threshold']:.3f}")